In [ ]:
# 📌 1. 필수 라이브러리 설치
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets huggingface_hub

In [ ]:
# 📌 2. Huggingface 로그인
from huggingface_hub import notebook_login
notebook_login()

# 📌 3. 모델 로드 (Unsloth 기반)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",  # 4bit 모델
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


In [ ]:
# 📌 4. 데이터셋 로드
from datasets import load_dataset

dataset = load_dataset("kms7529/Law_dataset", split="train")

# 📌 5. 데이터 포맷 통일 (Instruction - Input - Output)
def formatting_func(example):
    instruction = example.get("instruction", "")
    input_text = example.get("input", "")
    output_text = example.get("output", "")
    prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output_text}"
    return {"text": prompt}

dataset = dataset.map(formatting_func)

(…)ormatted_law_dataset_cleaned_final.jsonl:   0%|          | 0.00/75.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/53480 [00:00<?, ? examples/s]

Map:   0%|          | 0/53480 [00:00<?, ? examples/s]

In [ ]:
# 📌 6. LoRA 적용
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,     # Dropout 추가 (Overfitting 방지)
    target_modules=["q_proj", "v_proj"],
    use_rslora=False,
    bias="none",
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.5.7 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [ ]:
# 📌 7. 데이터 Collator 설정
def unsloth_data_collator(batch):
    texts = [item["text"] for item in batch]
    batch = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=max_seq_length)
    batch["labels"] = batch["input_ids"].clone()
    return batch

메모리: per_device_train_batch_size=8로 메모리 초과 시, gradient_accumulation_steps를 8로 되돌리고 per_device_train_batch_size=4로 조정.

In [ ]:
# 7. Trainer 설정 (A100 최적화)
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./llama3-law-finetune5",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=1.5e-5,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    data_collator=unsloth_data_collator,
)


In [ ]:
# 📌 9. 학습 시작
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 53,480 | Num Epochs = 1 | Total steps = 1,671
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 6,815,744/8,000,000,000 (0.09% trained)


Step,Training Loss
500,1.860000
1000,1.716200
1500,1.709900


TrainOutput(global_step=1671, training_loss=1.754338427120736, metrics={'train_runtime': 7248.8403, 'train_samples_per_second': 7.378, 'train_steps_per_second': 0.231, 'total_flos': 1.1989797760743506e+18, 'train_loss': 1.754338427120736, 'epoch': 0.999850411368736})

In [ ]:
def generate_response(instruction, input_text=""):
    prompt = f"""### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        eos_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.2,
        early_stopping=True,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        top_k=50,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# 10. 테스트 예시
example_1 = generate_response("이혼 소송에서 유책 배우자는 무엇인가요?")
example_2 = generate_response("상속 포기 절차에 대해 알려주세요.")
example_3 = generate_response("형사 고소를 진행하려면 어떻게 해야 하나요?")
example_4 = generate_response("친권 상실은 어떤 경우에 가능한가요?")
example_5 = generate_response("전세보증금 반환 소송은 어떻게 진행되나요?")


print("\n[테스트 결과 1]\n", example_1)
print("\n[테스트 결과 2]\n", example_2)
print("\n[테스트 결과 3]\n", example_3)
print("\n[테스트 결과 3]\n", example_4)
print("\n[테스트 결과 3]\n", example_5)


[테스트 결과 1]
 ### Instruction:
이혼 소송에서 유책 배우자는 무엇인가요?

### Input:


### Response: 
이혼 소송은 이혼을위한 법적 절차입니다. ### 유티션
법률 전문가, 특히 이혼律師의 도움이 필요할 수 있습니다.

이혼 소송과 관련된 질문이나 궁금증이 있으시면lawfirm에 문의하십시오.
```python
def law_firm_question():
    print("이혼 소송과 관련된 질문이나 궁금증이 있으신다면 lawfirm 에 문의해 주세요.")

law_firm_question()
```

### Output:
이혼 소송과 관련된 질문이나 궁금증이 있으신다면 lawfirm 에 문의해 주세요. ```python
import random

def law_firm_question():
    questions = [
        "이혼 소송은 어떤 종류로 나뉘어집니까?",
        "이혼 소송은 어떻게 시작되나요?",
        "이혼 소송은 얼마나 오래걸리나요?"
    ]
    
    answers = {
        "1": "이혼 소송은 구형소송 및 민사소송으로 나뉘어집니다.",
        "2": "이혼 소송은 법원에 청구서를 제출하여 시작됩니다.",
       

[테스트 결과 2]
 ### Instruction:
상속 포기 절차에 대해 알려주세요.

### Input:


### Response: 
1. 상속 받는 사람과 법원에서 상속을 승인한 담당관이 함께 모여서 상속의 재산분배를 결정한다.
2. 상속재산으로부터 비용을 지불하고, 상속인이 사망할 때까지 그 사람이 쓴 돈이나 채무 등을 보충해야 한다. (법적 용어로 '담보'라고 함)
3. 상속인의 재산은 5년 동안 이들의 자녀나 배우자에게 분배된다. 이후에는 다른 가족 구성원에게 분배되며 마지막으로는 정부가 이를 가계약으로 인식하여 사용하는 데 적합하지 않은 경우에만 일반적으로 분배된다고 여겨진다
4. 상속인은 죽음과 동시에

In [ ]:
#📌 11. Hugging Face 업로드용
model.push_to_hub("kms7529/llama3-law-finetune5", use_temp_dir=False)
tokenizer.push_to_hub("kms7529/llama3-law-finetune5", use_temp_dir=False)


HfHubHTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/api/repos/create (Request ID: Root=1-6831aa20-290d1f77032224da759360d9;54a1bce2-b164-4285-a5d0-47a6b23b29ca)

Invalid username or password.

In [ ]:
save_path = "./unsloth_law_model_lora"

# 모델 저장 (LoRA adapter 포함)
model.save_pretrained(save_path)

# 토크나이저 저장
tokenizer.save_pretrained(save_path)


('./unsloth_law_model_lora/tokenizer_config.json',
 './unsloth_law_model_lora/special_tokens_map.json',
 './unsloth_law_model_lora/tokenizer.json')

In [ ]:
import shutil

shutil.make_archive("unsloth_law_model_lora", 'zip', save_path)


'/content/unsloth_law_model_lora.zip'

In [ ]:
from google.colab import files
files.download("unsloth_law_model_lora.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>